# DSE Baselines — Notebook 1: PyTorch
MGPred, SDPred, MSSF using **UNIFIED SPLITS**.

**Split A** = Warm-start (pair-wise 10-fold). **Split B** = Drug cold-start (drug-level 10-fold).


In [1]:
%%time
REPO_URL = 'https://github.com/hungbuile04/DSE.git'
!git clone $REPO_URL
%cd DSE
!pip install -q numpy pandas scikit-learn torch
import os, torch
for split in ['A', 'B']:
    for m in ['MGPred', 'SDPred', 'MSSF']:
        os.makedirs(f'results/{m}_{split}', exist_ok=True)

# === Global speed optimizations ===
torch.backends.cudnn.benchmark = True              # auto-tune conv kernels
torch.set_float32_matmul_precision('medium')       # use TF32 on Ampere/Ada GPUs
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Mount Google Drive (for auto-saving results immediately after each model)
import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/DSE_results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'✅ Google Drive mounted at {DRIVE_DIR}')
except Exception as e:
    print(f'ℹ️ Google Drive mount skipped: {e}')


Cloning into 'DSE'...
remote: Enumerating objects: 282, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 282 (delta 27), reused 45 (delta 16), pack-reused 224 (from 1)
Receiving objects: 100% (282/282), 109.46 MiB | 51.39 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/content/DSE
GPU: NVIDIA L4
VRAM: 23.7 GB
Mounted at /content/drive
✅ Google Drive mounted at /content/drive/MyDrive/DSE_results
CPU times: user 3.66 s, sys: 1.12 s, total: 4.78 s
Wall time: 46.7 s


In [2]:
SPLIT_TYPE = 'A'  # 'A' = Warm-start (pair-wise), 'B' = Drug cold-start

In [3]:
import sys
import os
import pickle
import argparse
import numpy as np
sys.path.insert(0, '/content/DSE/shared_data')
from split_adapter import load_splits

---
## 1. MGPred
- `train_test()` returns `(rmse, mae)` and saves best predictions to `final_p{fold}.p`
- Data format: `(SE_id, drug_id, freq)` — reversed order!

In [ ]:
%%time
%cd /content/DSE/MGPred

# --- Speed patches (sed) ---
!sed -i 's/num_workers=16/num_workers=4/g' Ten_Fold_test.py
!sed -i 's/pin_memory=True)/pin_memory=True, persistent_workers=True)/g' Ten_Fold_test.py
!sed -i 's/loss.backward(retain_graph = True)/loss.backward()/g' Ten_Fold_test.py
!sed -i '/rmse1, mae1, ground_i, ground_u, ground_truth, pred = test(model, _train, _train, device)/d' Ten_Fold_test.py
!sed -i '/<Train> RMSE/d' Ten_Fold_test.py

# --- KEY: Test every 3 epochs only (Python patch) ---
import re
with open('Ten_Fold_test.py', 'r') as f:
    code = f.read()

# Wrap test call: only run every 3 epochs or last epoch
code = code.replace(
    '        rmse, mae, ground_i, ground_u, ground_truth, pred = test(model, _test, _test, device)\r\n\r\n        if rmse_mn > rmse:',
    '        if epoch % 3 == 0 or epoch >= args.epochs - 1:\r\n            rmse, mae, ground_i, ground_u, ground_truth, pred = test(model, _test, _test, device)\r\n        else:\r\n            print(f"  [epoch {epoch}] train done, skip eval")\r\n            continue\r\n\r\n        if rmse_mn > rmse:'
)
code = code.replace('endure_count > 30', 'endure_count > 2')
with open('Ten_Fold_test.py', 'w') as f:
    f.write(code)
print("Patched: test every 3 epochs")

sys.path.insert(0, '/content/DSE/MGPred')
from Ten_Fold_test import train_test, Extract_positive_negative_samples

with open('data/drug_side.pkl', 'rb') as f:
    drug_side = pickle.load(f)

addition_neg, _, final_neg = Extract_positive_negative_samples(drug_side, addition_negative_number='all')
addition_neg = np.vstack((addition_neg, final_neg))
data_neg = [(row[1], row[0], row[2]) for row in addition_neg]

args = argparse.Namespace(
    epochs=15, lr=0.001, embed_dim=64, weight_decay=0.0005,
    N=30000, droprate=0.5, batch_size=2048, test_batch_size=2048,
    rawpath='/content/DSE/MGPred/data', dataset=''
)

mgpred_results = []
for fold in range(10):
    print(f'\n===== MGPred Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, SPLIT_TYPE)
    data_train = [(int(r[1]), int(r[0]), r[2]) for r in train_pos]
    data_test = [(int(r[1]), int(r[0]), r[2]) for r in test_pos]

    rmse, mae = train_test(data_train, data_test, data_neg, fold+1, args)
    mgpred_results.append((rmse, mae))

    with open(f'final_p{fold+1}.p', 'rb') as f:
        ground_i, ground_u, ground_truth, pred = pickle.load(f)
    np.save(f'/content/DSE/results/MGPred_A/fold_{fold}_labels.npy', ground_truth)
    np.save(f'/content/DSE/results/MGPred_A/fold_{fold}_preds.npy', pred)
    if os.path.exists('/content/drive/MyDrive'):
        dst_dir = f'{DRIVE_DIR}/MGPred_A'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(f'/content/DSE/results/MGPred_A/fold_{fold}_labels.npy', f'{dst_dir}/fold_{fold}_labels.npy')
        shutil.copy(f'/content/DSE/results/MGPred_A/fold_{fold}_preds.npy', f'{dst_dir}/fold_{fold}_preds.npy')

print(f'\n=== MGPred Final ===')
print(f'RMSE: {np.mean([r[0] for r in mgpred_results]):.4f} +/- {np.std([r[0] for r in mgpred_results]):.4f}')
print(f'MAE:  {np.mean([r[1] for r in mgpred_results]):.4f} +/- {np.std([r[1] for r in mgpred_results]):.4f}')


if os.path.exists('/content/drive/MyDrive'):
    print(f'💾 MGPred_A results auto-saved to Google Drive: {DRIVE_DIR}/MGPred_A')

---
## 2. SDPred
- `train_test()` returns only `(auc, aupr, rmse, mae)` — 4 values
- Need to patch return statement to also return predictions
- ConvNCF(7570) is in `up_ten_fold.py`, not `model.py`
- Data format: `(drug_id, se_id, freq)` — normal order

In [ ]:
%%time
%cd /content/DSE
!cp -f shared_data/sdpred_750/* SDPred/data/
%cd /content/DSE/SDPred

# Patch dims: ConvNCF is in up_ten_fold.py line ~150
!sed -i 's/ConvNCF(7570,/ConvNCF(7500,/g' up_ten_fold.py
!sed -i 's/num_workers=16/num_workers=4/g' up_ten_fold.py
!sed -i 's/pin_memory=True)/pin_memory=True, persistent_workers=True)/g' up_ten_fold.py
!sed -i 's/CUDA_VISIBLE_DEVICES"\] = "2"/CUDA_VISIBLE_DEVICES"] = "0"/g' up_ten_fold.py
!sed -i 's/endure_count > 10/endure_count > 5/g' up_ten_fold.py
!sed -i 's/dropout1=0.8, dropout2=0.8/dropout1=0.5, dropout2=0.5/g' network.py

# CRITICAL: Patch train_test to return predictions
# Original line 198: return i_auc, iPR_auc, rmse, mae
!sed -i 's/return i_auc, iPR_auc, rmse, mae$/return i_auc, iPR_auc, rmse, mae, ground_truth, pred1, pred2/' up_ten_fold.py

# Fix: cast float indices to int (numpy mixed-type array issue)
!sed -i 's/drug_side\[data_test\[i, 0\], data_test\[i, 1\]\]/drug_side[int(data_test[i, 0]), int(data_test[i, 1])]/' up_ten_fold.py
!sed -i 's/data_test\[:, 0\]]/data_test[:, 0].astype(int)]/' up_ten_fold.py
!sed -i 's/data_test\[:, 1\]]/data_test[:, 1].astype(int)]/' up_ten_fold.py
!sed -i 's/data_train\[:, 0\]]/data_train[:, 0].astype(int)]/' up_ten_fold.py
!sed -i 's/data_train\[:, 1\]]/data_train[:, 1].astype(int)]/' up_ten_fold.py

sys.path.insert(0, '/content/DSE/SDPred')
from up_ten_fold import train_test as sdpred_train_test
from up_ten_fold import Extract_positive_negative_samples as sdpred_extract

with open('data/drug_side.pkl', 'rb') as f:
    drug_side_sd = pickle.load(f)

add_neg, pos, neg = sdpred_extract(drug_side_sd, 'all')
all_data_neg = [(r[0], r[1], r[2]) for r in np.vstack((add_neg, neg))]
zero_pairs = np.argwhere(drug_side_sd == 0)

args_sd = argparse.Namespace(
    epochs=100, lr=0.0001, embed_dim=128, weight_decay=1e-5,
    N=30000, droprate=0.5, batch_size=512, test_batch_size=512,
    rawpath='/content/DSE/SDPred/data'
)

sdpred_results = []
for fold in range(10):
    print(f'\n===== SDPred Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, SPLIT_TYPE)

    np.random.seed(42 + fold)
    shuffled_zeros = np.random.permutation(len(zero_pairs))
    train_neg_idx = shuffled_zeros[:len(train_pos)]
    test_neg_idx = shuffled_zeros[len(train_pos):len(train_pos)+len(test_pos)]

    train_neg = [(int(zero_pairs[i][0]), int(zero_pairs[i][1]), 0.0) for i in train_neg_idx]
    test_neg = [(int(zero_pairs[i][0]), int(zero_pairs[i][1]), 0.0) for i in test_neg_idx]

    data_train = [(int(r[0]), int(r[1]), r[2]) for r in train_pos] + train_neg
    data_test = [(int(r[0]), int(r[1]), r[2]) for r in test_pos] + test_neg

    i_auc, iPR_auc, rmse, mae, ground_truth, pred1, pred2 = sdpred_train_test(
        data_train, data_test, all_data_neg, fold+1, args_sd)
    sdpred_results.append((i_auc, iPR_auc, rmse, mae))

    np.save(f'/content/DSE/results/SDPred_A/fold_{fold}_labels.npy', ground_truth)
    np.save(f'/content/DSE/results/SDPred_A/fold_{fold}_preds.npy', pred2)
    if os.path.exists('/content/drive/MyDrive'):
        dst_dir = f'{DRIVE_DIR}/SDPred_A'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(f'/content/DSE/results/SDPred_A/fold_{fold}_labels.npy', f'{dst_dir}/fold_{fold}_labels.npy')
        shutil.copy(f'/content/DSE/results/SDPred_A/fold_{fold}_preds.npy', f'{dst_dir}/fold_{fold}_preds.npy')

print(f'\n=== SDPred Final ===')
print(f'AUC:  {np.mean([r[0] for r in sdpred_results]):.4f}')
print(f'AUPR: {np.mean([r[1] for r in sdpred_results]):.4f}')
print(f'RMSE: {np.mean([r[2] for r in sdpred_results]):.4f}')
print(f'MAE:  {np.mean([r[3] for r in sdpred_results]):.4f}')

if os.path.exists('/content/drive/MyDrive'):
    print(f'💾 SDPred_A results auto-saved to Google Drive: {DRIVE_DIR}/SDPred_A')

/content/DSE
/content/DSE/SDPred


<timed exec>:29: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.



===== SDPred Fold 1/10 =====
Epoch: 1 <Train> RMSE: 1.15367, MAE: 0.89192, AUC: 0.90577, AUPR: 0.90747 
Epoch: 2 <Train> RMSE: 0.88524, MAE: 0.69090, AUC: 0.90595, AUPR: 0.90830 
Epoch: 3 <Train> RMSE: 0.91740, MAE: 0.72832, AUC: 0.91159, AUPR: 0.91343 
Epoch: 4 <Train> RMSE: 0.85777, MAE: 0.67675, AUC: 0.91605, AUPR: 0.91738 
Epoch: 5 <Train> RMSE: 0.86252, MAE: 0.68510, AUC: 0.91831, AUPR: 0.91936 
Epoch: 6 <Train> RMSE: 0.83217, MAE: 0.65732, AUC: 0.91947, AUPR: 0.92061 
Epoch: 7 <Train> RMSE: 0.75311, MAE: 0.58387, AUC: 0.92060, AUPR: 0.92142 
Epoch: 8 <Train> RMSE: 0.70625, MAE: 0.54103, AUC: 0.92118, AUPR: 0.92186 
Epoch: 9 <Train> RMSE: 0.73280, MAE: 0.56559, AUC: 0.92259, AUPR: 0.92286 
Epoch: 10 <Train> RMSE: 0.69936, MAE: 0.53431, AUC: 0.92352, AUPR: 0.92385 
Epoch: 11 <Train> RMSE: 0.76099, MAE: 0.59182, AUC: 0.92381, AUPR: 0.92445 
Epoch: 12 <Train> RMSE: 0.72415, MAE: 0.55987, AUC: 0.92494, AUPR: 0.92559 
Epoch: 13 <Train> RMSE: 0.68847, MAE: 0.52507, AUC: 0.92522, AUPR: 

---
## 3. MSSF
- `train_test(data_train, data_test, args)` — NO fold argument!
- Returns 8 metrics only, need to patch to also return predictions
- Multi-class: class 0-4 → frequency 1-5

In [4]:
%%time
%cd /content/DSE
!cp -f shared_data/mssf_750/* MSSF/Datas/
%cd /content/DSE/MSSF

# Patch dims: 757 -> 750
!sed -i 's/757\*11/750*11/g' model.py
!sed -i 's/drugs_inputdim=757/drugs_inputdim=750/g' model.py
!sed -i 's/drug_inputdim=757/drug_inputdim=750/g' model.py
!sed -i 's/num_workers=16/num_workers=4/g' mssf.py
!sed -i 's/CUDA_VISIBLE_DEVICES"\] = "3"/CUDA_VISIBLE_DEVICES"] = "0"/g' mssf.py

# CRITICAL: Patch train_test to return raw predictions
# Original: return acc_tested,wf1_tested,maf1_tested,ka_tested,mcc_tested,maprec_tested,mareca_tested,maaupr_tested
# We need rating_te (ground truth) and pred_te (predictions) which exist in the function
!sed -i 's/return acc_tested,wf1_tested,maf1_tested,ka_tested,mcc_tested,maprec_tested,mareca_tested,maaupr_tested/return acc_tested,wf1_tested,maf1_tested,ka_tested,mcc_tested,maprec_tested,mareca_tested,maaupr_tested,rating_te_best,pred_te_best/' mssf.py
# Also need to save best predictions: add tracking variables
!sed -i '/acc_tested = 0/a\    rating_te_best = None\n    pred_te_best = None' mssf.py
!sed -i '/acc_tested = acc_te/a\            rating_te_best = rating_te\n            pred_te_best = pred_te' mssf.py

# Fix: numpy arrays from mixed int/float tuples become all float64 → cast indices to int
!sed -i 's/drug_side\[data_test\[i, 0\], data_test\[i, 1\]\]/drug_side[int(data_test[i, 0]), int(data_test[i, 1])]/' mssf.py
!sed -i 's/data_test\[:, 0\]]/data_test[:, 0].astype(int)]/' mssf.py
!sed -i 's/data_test\[:, 1\]]/data_test[:, 1].astype(int)]/' mssf.py
!sed -i 's/data_train\[:, 0\]]/data_train[:, 0].astype(int)]/' mssf.py
!sed -i 's/data_train\[:, 1\]]/data_train[:, 1].astype(int)]/' mssf.py

sys.path.insert(0, '/content/DSE/MSSF')
from mssf import train_test as mssf_train_test

args_mssf = argparse.Namespace(
    epochs=50, lr=0.0001, embed_dim=128, weight_decay=1e-5,
    dropout=0.4, gp=64, batch_size=512, test_batch_size=512,
    rawpath='/content/DSE/MSSF/Datas'
)

mssf_results = []
for fold in range(10):
    print(f'\n===== MSSF Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, SPLIT_TYPE)
    data_train = [(int(r[0]), int(r[1]), r[2]) for r in train_pos]
    data_test = [(int(r[0]), int(r[1]), r[2]) for r in test_pos]

    # MSSF: train_test(data_train, data_test, args) — NO fold arg
    acc, wf1, maf1, kappa, mcc, prec, recall, aupr, labels, preds = mssf_train_test(
        data_train, data_test, args_mssf)
    mssf_results.append((acc, wf1, maf1, kappa))

    # Class 0-4 → frequency 1-5
    np.save(f'/content/DSE/results/MSSF_A/fold_{fold}_labels.npy', np.array(labels) + 1)
    np.save(f'/content/DSE/results/MSSF_A/fold_{fold}_preds.npy', np.array(preds) + 1)
    if os.path.exists('/content/drive/MyDrive'):
        dst_dir = f'{DRIVE_DIR}/MSSF_A'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(f'/content/DSE/results/MSSF_A/fold_{fold}_labels.npy', f'{dst_dir}/fold_{fold}_labels.npy')
        shutil.copy(f'/content/DSE/results/MSSF_A/fold_{fold}_preds.npy', f'{dst_dir}/fold_{fold}_preds.npy')

print(f'\n=== MSSF Final ===')
print(f'Acc:   {np.mean([r[0] for r in mssf_results]):.4f}')
print(f'WF1:   {np.mean([r[1] for r in mssf_results]):.4f}')
print(f'MacF1: {np.mean([r[2] for r in mssf_results]):.4f}')
print(f'Kappa: {np.mean([r[3] for r in mssf_results]):.4f}')

if os.path.exists('/content/drive/MyDrive'):
    print(f'💾 MSSF_A results auto-saved to Google Drive: {DRIVE_DIR}/MSSF_A')

/content/DSE
/content/DSE/MSSF

===== MSSF Fold 1/10 =====
Epoch: 1 <Train> acc: 0.60088, weighted_f1: 0.54907, macro_f1: 0.36545, kappa: 0.34226 ,mcc: 0.36269,precision:0.55450,recall: 0.36335,aupr:0.50233
Epoch: 1 <Test> acc: 0.59439, weighted_f1: 0.54155, macro_f1: 0.35414, kappa: 0.33100 ,mcc: 0.35108,precision:0.58251,recall: 0.35554,aupr:0.48853
Epoch: 2 <Train> acc: 0.62866, weighted_f1: 0.58043, macro_f1: 0.42481, kappa: 0.39572 ,mcc: 0.41265,precision:0.65076,recall: 0.40424,aupr:0.56993
Epoch: 2 <Test> acc: 0.62379, weighted_f1: 0.57593, macro_f1: 0.41510, kappa: 0.38736 ,mcc: 0.40407,precision:0.64271,recall: 0.39565,aupr:0.55604
Epoch: 3 <Train> acc: 0.63639, weighted_f1: 0.61695, macro_f1: 0.50696, kappa: 0.44155 ,mcc: 0.45266,precision:0.66970,recall: 0.47609,aupr:0.61749
Epoch: 3 <Test> acc: 0.61030, weighted_f1: 0.59311, macro_f1: 0.48681, kappa: 0.40177 ,mcc: 0.41254,precision:0.64770,recall: 0.45379,aupr:0.58834
Epoch: 4 <Train> acc: 0.57519, weighted_f1: 0.49523, mac

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 1 <Train> acc: 0.57247, weighted_f1: 0.52911, macro_f1: 0.34702, kappa: 0.32793 ,mcc: 0.33770,precision:0.43276,recall: 0.36923,aupr:0.48399
Epoch: 1 <Test> acc: 0.57405, weighted_f1: 0.53103, macro_f1: 0.34039, kappa: 0.33343 ,mcc: 0.34350,precision:0.46803,recall: 0.36800,aupr:0.46430
Epoch: 2 <Train> acc: 0.64210, weighted_f1: 0.62012, macro_f1: 0.52688, kappa: 0.44006 ,mcc: 0.44796,precision:0.61322,recall: 0.50595,aupr:0.59230
Epoch: 2 <Test> acc: 0.63286, weighted_f1: 0.60702, macro_f1: 0.49734, kappa: 0.42454 ,mcc: 0.43287,precision:0.58903,recall: 0.48057,aupr:0.55163
Epoch: 3 <Train> acc: 0.65346, weighted_f1: 0.64057, macro_f1: 0.55843, kappa: 0.47050 ,mcc: 0.47767,precision:0.58381,recall: 0.58299,aupr:0.62041
Epoch: 3 <Test> acc: 0.63771, weighted_f1: 0.62494, macro_f1: 0.53313, kappa: 0.44783 ,mcc: 0.45468,precision:0.55752,recall: 0.55618,aupr:0.58789
Epoch: 4 <Train> acc: 0.55101, weighted_f1: 0.55180, macro_f1: 0.52798, kappa: 0.38687 ,mcc: 0.40862,precision:0.53

---
## Split B: Drug Cold-start

Re-run all 3 models with `SPLIT_TYPE = 'B'`.

> **Note**: SDPred and MSSF use drug-SE profiles as features. In cold-start, test drugs have
> no known associations — the unified split adapter handles this by zeroing out test drug rows
> in the similarity matrices computed inside `read_raw_data()`.


In [ ]:
%%time
# === MGPred Split B ===
%cd /content/DSE/MGPred

mgpred_b_results = []
for fold in range(10):
    print(f'\n===== MGPred Split B Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, 'B')
    data_train = [(int(r[1]), int(r[0]), r[2]) for r in train_pos]
    data_test = [(int(r[1]), int(r[0]), r[2]) for r in test_pos]

    rmse, mae = train_test(data_train, data_test, data_neg, fold+1, args)
    mgpred_b_results.append((rmse, mae))

    with open(f'final_p{fold+1}.p', 'rb') as f:
        ground_i, ground_u, ground_truth, pred = pickle.load(f)
    np.save(f'/content/DSE/results/MGPred_B/fold_{fold}_labels.npy', ground_truth)
    np.save(f'/content/DSE/results/MGPred_B/fold_{fold}_preds.npy', pred)
    if os.path.exists('/content/drive/MyDrive'):
        dst_dir = f'{DRIVE_DIR}/MGPred_B'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(f'/content/DSE/results/MGPred_B/fold_{fold}_labels.npy', f'{dst_dir}/fold_{fold}_labels.npy')
        shutil.copy(f'/content/DSE/results/MGPred_B/fold_{fold}_preds.npy', f'{dst_dir}/fold_{fold}_preds.npy')

print(f'\n=== MGPred Split B Final ===')
print(f'RMSE: {np.mean([r[0] for r in mgpred_b_results]):.4f} ± {np.std([r[0] for r in mgpred_b_results]):.4f}')
print(f'MAE:  {np.mean([r[1] for r in mgpred_b_results]):.4f} ± {np.std([r[1] for r in mgpred_b_results]):.4f}')


if os.path.exists('/content/drive/MyDrive'):
    print(f'💾 MGPred_B results auto-saved to Google Drive: {DRIVE_DIR}/MGPred_B')

In [ ]:
%%time
# === SDPred Split B ===
%cd /content/DSE/SDPred

sdpred_b_results = []
for fold in range(10):
    print(f'\n===== SDPred Split B Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, 'B')

    np.random.seed(42 + fold)
    shuffled_zeros = np.random.permutation(len(zero_pairs))
    train_neg_idx = shuffled_zeros[:len(train_pos)]
    test_neg_idx = shuffled_zeros[len(train_pos):len(train_pos)+len(test_pos)]

    train_neg = [(int(zero_pairs[i][0]), int(zero_pairs[i][1]), 0.0) for i in train_neg_idx]
    test_neg = [(int(zero_pairs[i][0]), int(zero_pairs[i][1]), 0.0) for i in test_neg_idx]

    data_train = [(int(r[0]), int(r[1]), r[2]) for r in train_pos] + train_neg
    data_test = [(int(r[0]), int(r[1]), r[2]) for r in test_pos] + test_neg

    i_auc, iPR_auc, rmse, mae, ground_truth, pred1, pred2 = sdpred_train_test(
        data_train, data_test, all_data_neg, fold+1, args_sd)
    sdpred_b_results.append((i_auc, iPR_auc, rmse, mae))

    np.save(f'/content/DSE/results/SDPred_B/fold_{fold}_labels.npy', ground_truth)
    np.save(f'/content/DSE/results/SDPred_B/fold_{fold}_preds.npy', pred2)
    if os.path.exists('/content/drive/MyDrive'):
        dst_dir = f'{DRIVE_DIR}/SDPred_B'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(f'/content/DSE/results/SDPred_B/fold_{fold}_labels.npy', f'{dst_dir}/fold_{fold}_labels.npy')
        shutil.copy(f'/content/DSE/results/SDPred_B/fold_{fold}_preds.npy', f'{dst_dir}/fold_{fold}_preds.npy')

print(f'\n=== SDPred Split B Final ===')
print(f'AUC:  {np.mean([r[0] for r in sdpred_b_results]):.4f}')
print(f'AUPR: {np.mean([r[1] for r in sdpred_b_results]):.4f}')
print(f'RMSE: {np.mean([r[2] for r in sdpred_b_results]):.4f}')
print(f'MAE:  {np.mean([r[3] for r in sdpred_b_results]):.4f}')


if os.path.exists('/content/drive/MyDrive'):
    print(f'💾 SDPred_B results auto-saved to Google Drive: {DRIVE_DIR}/SDPred_B')

In [5]:
%%time
# === MSSF Split B ===
%cd /content/DSE/MSSF

mssf_b_results = []
for fold in range(10):
    print(f'\n===== MSSF Split B Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, 'B')
    data_train = [(int(r[0]), int(r[1]), r[2]) for r in train_pos]
    data_test = [(int(r[0]), int(r[1]), r[2]) for r in test_pos]

    acc, wf1, maf1, kappa, mcc, prec, recall, aupr, labels, preds = mssf_train_test(
        data_train, data_test, args_mssf)
    mssf_b_results.append((acc, wf1, maf1, kappa))

    np.save(f'/content/DSE/results/MSSF_B/fold_{fold}_labels.npy', np.array(labels) + 1)
    np.save(f'/content/DSE/results/MSSF_B/fold_{fold}_preds.npy', np.array(preds) + 1)
    if os.path.exists('/content/drive/MyDrive'):
        dst_dir = f'{DRIVE_DIR}/MSSF_B'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(f'/content/DSE/results/MSSF_B/fold_{fold}_labels.npy', f'{dst_dir}/fold_{fold}_labels.npy')
        shutil.copy(f'/content/DSE/results/MSSF_B/fold_{fold}_preds.npy', f'{dst_dir}/fold_{fold}_preds.npy')

print(f'\n=== MSSF Split B Final ===')
print(f'Acc:   {np.mean([r[0] for r in mssf_b_results]):.4f}')
print(f'WF1:   {np.mean([r[1] for r in mssf_b_results]):.4f}')
print(f'MacF1: {np.mean([r[2] for r in mssf_b_results]):.4f}')
print(f'Kappa: {np.mean([r[3] for r in mssf_b_results]):.4f}')


if os.path.exists('/content/drive/MyDrive'):
    print(f'💾 MSSF_B results auto-saved to Google Drive: {DRIVE_DIR}/MSSF_B')

/content/DSE/MSSF

===== MSSF Split B Fold 1/10 =====
Epoch: 1 <Train> acc: 0.56830, weighted_f1: 0.52931, macro_f1: 0.37423, kappa: 0.32876 ,mcc: 0.33696,precision:0.41657,recall: 0.39189,aupr:0.47400
Epoch: 1 <Test> acc: 0.50130, weighted_f1: 0.44897, macro_f1: 0.31240, kappa: 0.17383 ,mcc: 0.19444,precision:0.35907,recall: 0.34202,aupr:0.34818
Epoch: 2 <Train> acc: 0.61431, weighted_f1: 0.61036, macro_f1: 0.52614, kappa: 0.42713 ,mcc: 0.42906,precision:0.54876,recall: 0.51580,aupr:0.58440
Epoch: 2 <Test> acc: 0.45460, weighted_f1: 0.37545, macro_f1: 0.26967, kappa: 0.09660 ,mcc: 0.11469,precision:0.36955,recall: 0.30031,aupr:0.32632
Epoch: 3 <Train> acc: 0.60586, weighted_f1: 0.59688, macro_f1: 0.50228, kappa: 0.41487 ,mcc: 0.42776,precision:0.58680,recall: 0.47970,aupr:0.60024
Epoch: 3 <Test> acc: 0.50620, weighted_f1: 0.45965, macro_f1: 0.33254, kappa: 0.16892 ,mcc: 0.18758,precision:0.41880,recall: 0.31959,aupr:0.36841
Epoch: 4 <Train> acc: 0.53958, weighted_f1: 0.55778, macro_f1

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 31 <Train> acc: 0.70737, weighted_f1: 0.68216, macro_f1: 0.67325, kappa: 0.52789 ,mcc: 0.56828,precision:0.77778,recall: 0.63745,aupr:0.83585
Epoch: 31 <Test> acc: 0.47737, weighted_f1: 0.32809, macro_f1: 0.16556, kappa: 0.00845 ,mcc: 0.02184,precision:0.20985,recall: 0.21494,aupr:0.24942
Epoch: 32 <Train> acc: 0.79596, weighted_f1: 0.79915, macro_f1: 0.79253, kappa: 0.71130 ,mcc: 0.71818,precision:0.77552,recall: 0.82672,aupr:0.89260
Epoch: 32 <Test> acc: 0.41107, weighted_f1: 0.39371, macro_f1: 0.30252, kappa: 0.11723 ,mcc: 0.12432,precision:0.34522,recall: 0.32479,aupr:0.29954
Epoch: 33 <Train> acc: 0.79421, weighted_f1: 0.79737, macro_f1: 0.78450, kappa: 0.70552 ,mcc: 0.71043,precision:0.77939,recall: 0.81412,aupr:0.89249
Epoch: 33 <Test> acc: 0.42894, weighted_f1: 0.38037, macro_f1: 0.28572, kappa: 0.10046 ,mcc: 0.11171,precision:0.40742,recall: 0.30394,aupr:0.30846
Epoch: 34 <Train> acc: 0.80204, weighted_f1: 0.80482, macro_f1: 0.80033, kappa: 0.72051 ,mcc: 0.72681,precisi

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 31 <Train> acc: 0.75708, weighted_f1: 0.74369, macro_f1: 0.68754, kappa: 0.62001 ,mcc: 0.63244,precision:0.80539,recall: 0.63452,aupr:0.82319
Epoch: 31 <Test> acc: 0.50442, weighted_f1: 0.35430, macro_f1: 0.16228, kappa: 0.01660 ,mcc: 0.04378,precision:0.35897,recall: 0.21346,aupr:0.27262
Epoch: 32 <Train> acc: 0.79955, weighted_f1: 0.80051, macro_f1: 0.77039, kappa: 0.71436 ,mcc: 0.71946,precision:0.73781,recall: 0.83618,aupr:0.88935
Epoch: 32 <Test> acc: 0.49610, weighted_f1: 0.39757, macro_f1: 0.24154, kappa: 0.07978 ,mcc: 0.10783,precision:0.38032,recall: 0.26121,aupr:0.29575
Epoch: 33 <Train> acc: 0.74528, weighted_f1: 0.74646, macro_f1: 0.74428, kappa: 0.64942 ,mcc: 0.66705,precision:0.71273,recall: 0.81424,aupr:0.86523
Epoch: 33 <Test> acc: 0.49974, weighted_f1: 0.46906, macro_f1: 0.30679, kappa: 0.14688 ,mcc: 0.15188,precision:0.40512,recall: 0.29155,aupr:0.31745
Epoch: 34 <Train> acc: 0.67530, weighted_f1: 0.67246, macro_f1: 0.66991, kappa: 0.57374 ,mcc: 0.60643,precisi

...

Epoch: 2 <Train> acc: 0.59561, weighted_f1: 0.58850, macro_f1: 0.43310, kappa: 0.40545 ,mcc: 0.40928,precision:0.51574,recall: 0.46366,aupr:0.53019
Epoch: 2 <Test> acc: 0.51750, weighted_f1: 0.44375, macro_f1: 0.30985, kappa: 0.25017 ,mcc: 0.28265,precision:0.37621,recall: 0.33861,aupr:0.44214
Epoch: 3 <Train> acc: 0.64433, weighted_f1: 0.61302, macro_f1: 0.48482, kappa: 0.43691 ,mcc: 0.45578,precision:0.63464,recall: 0.47927,aupr:0.61173
Epoch: 3 <Test> acc: 0.48871, weighted_f1: 0.37875, macro_f1: 0.24675, kappa: 0.14156 ,mcc: 0.20994,precision:0.55009,recall: 0.27395,aupr:0.39771


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 4 <Train> acc: 0.66234, weighted_f1: 0.64553, macro_f1: 0.52963, kappa: 0.46597 ,mcc: 0.47319,precision:0.67176,recall: 0.48781,aupr:0.64350
Epoch: 4 <Test> acc: 0.46960, weighted_f1: 0.34736, macro_f1: 0.19071, kappa: 0.08777 ,mcc: 0.14595,precision:0.36089,recall: 0.23352,aupr:0.38875


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 5 <Train> acc: 0.65813, weighted_f1: 0.61105, macro_f1: 0.46251, kappa: 0.44861 ,mcc: 0.46288,precision:0.70919,recall: 0.44926,aupr:0.63477
Epoch: 5 <Test> acc: 0.54133, weighted_f1: 0.46177, macro_f1: 0.27889, kappa: 0.25149 ,mcc: 0.28796,precision:0.36105,recall: 0.30110,aupr:0.40350
Epoch: 6 <Train> acc: 0.68661, weighted_f1: 0.65784, macro_f1: 0.55172, kappa: 0.49547 ,mcc: 0.51028,precision:0.72300,recall: 0.51239,aupr:0.69383
Epoch: 6 <Test> acc: 0.53189, weighted_f1: 0.45111, macro_f1: 0.30050, kappa: 0.22977 ,mcc: 0.28553,precision:0.49425,recall: 0.31163,aupr:0.42021
Epoch: 7 <Train> acc: 0.68749, weighted_f1: 0.68874, macro_f1: 0.63645, kappa: 0.54140 ,mcc: 0.54642,precision:0.64832,recall: 0.63618,aupr:0.70471
Epoch: 7 <Test> acc: 0.54157, weighted_f1: 0.52130, macro_f1: 0.40059, kappa: 0.31055 ,mcc: 0.31641,precision:0.45192,recall: 0.38460,aupr:0.42215
Epoch: 8 <Train> acc: 0.66328, weighted_f1: 0.64257, macro_f1: 0.60515, kappa: 0.48837 ,mcc: 0.50451,precision:0.64

---
## Unified Evaluation

In [ ]:
%cd /content/DSE
!python shared_data/unified_eval.py --results_dir ./results